In [ ]:
#| default_exp handlers.pipeline.contracts

# Contracts

`HandlerConfig` and its supporting Pydantic schema models are the canonical runtime contract for the GeneralHandler pipeline.

In [ ]:
#| export
from __future__ import annotations
import importlib
import importlib.util
from pathlib import Path
from typing import Annotated, Any, Optional, Union
from pydantic import BaseModel, Field, ConfigDict, TypeAdapter, computed_field
from marisco.configs import NC_GLOBAL_ATTRS

## Contract Models

In [ ]:
#| export
# Modules scanned (in order) when spec.name is used; first hit wins.
_SHARED_SCAN_MODULES = [
    "marisco.callbacks.shared",
    "marisco.callbacks.core",
]


class _HandlerSection(BaseModel):
    module_name: str
    title: str = ""
    description: str = ""


class _DataSourceSection(BaseModel):
    url: str
    fname_out: str
    zenodo_id: str = ""
    format: str = "csv"


class _RenameColsSection(BaseModel):
    mapping: dict[str, str] = Field(default_factory=dict)
    string_cast: list[str] = Field(default_factory=list)


class _ParseDateTimeSection(BaseModel):
    col_date: Optional[str] = None
    col_time: Optional[str] = None
    format: str = "%Y-%m-%d"


class _MeltSection(BaseModel):
    meta_cols: list[str] = Field(default_factory=list)
    spec: list[dict[str, Any]] = Field(default_factory=list)


class _NomenclaturesSection(BaseModel):
    nuclide_lut: dict[str, int] = Field(default_factory=dict)
    unit_lut: dict[str, int] = Field(default_factory=dict)
    lab_lut: dict[str, int] = Field(default_factory=dict)
    lab_constants: dict[str, str] = Field(default_factory=dict)
    area_default: int = 0


class _OutputSection(BaseModel):
    keywords: list[str] = Field(default_factory=list)
    global_attrs: dict[str, str] = Field(default_factory=dict)


class _RawHandlerContract(BaseModel):
    handler: _HandlerSection
    data_source: _DataSourceSection
    rename_cols: _RenameColsSection = Field(default_factory=_RenameColsSection)
    columns: dict[str, str] = Field(default_factory=dict)
    normalize_case: dict[str, str] = Field(default_factory=dict)
    parse_datetime: _ParseDateTimeSection = Field(default_factory=_ParseDateTimeSection)
    time_format: Optional[str] = None
    melt: _MeltSection = Field(default_factory=_MeltSection)
    unit_conversions: list[dict[str, Any]] = Field(default_factory=list)
    nomenclatures: _NomenclaturesSection = Field(default_factory=_NomenclaturesSection)
    output: _OutputSection = Field(default_factory=_OutputSection)
    loader: Optional[dict[str, Any]] = None


class BasePluginSpec(BaseModel):
    "External Callback injection spec with polymorphic resolution."
    model_config = ConfigDict(populate_by_name=True)
    path: Optional[str] = None
    name: Optional[str] = None
    file: Optional[str] = None
    class_: Optional[str] = Field(None, alias="class")
    args: dict = Field(default_factory=dict)

    def resolve(self, yaml_dir: Path = None):
        raise NotImplementedError

    def resolve_fn(self, yaml_dir: Path = None):
        raise ValueError("Custom loader PluginSpec must use 'path:' (function, not class).")


class LegacyPathPluginSpec(BasePluginSpec):
    "Legacy fully-qualified dotted import path."
    path: str

    def resolve(self, yaml_dir: Path = None):
        module_path, class_name = self.path.rsplit(".", 1)
        return getattr(importlib.import_module(module_path), class_name)

    def resolve_fn(self, yaml_dir: Path = None):
        module_path, fn_name = self.path.rsplit(".", 1)
        return getattr(importlib.import_module(module_path), fn_name)


class NamePluginSpec(BasePluginSpec):
    "Shorthand callback name auto-resolved from shared/core modules."
    name: str

    def resolve(self, yaml_dir: Path = None):
        for mod_path in _SHARED_SCAN_MODULES:
            mod = importlib.import_module(mod_path)
            if hasattr(mod, self.name):
                return getattr(mod, self.name)
        raise ImportError(
            f"Callback '{self.name}' not found in {_SHARED_SCAN_MODULES}. "
            "Use the full 'path:' form to specify a non-shared callback."
        )


class FilePluginSpec(BasePluginSpec):
    "Local file import relative to the YAML directory."
    file: str
    class_: str = Field(alias="class")

    def resolve(self, yaml_dir: Path = None):
        if yaml_dir is None:
            raise ValueError("yaml_dir is required for file-based plugin loading")
        file_path = (Path(yaml_dir) / self.file).resolve()
        if not file_path.exists():
            raise FileNotFoundError(f"Plugin file not found: {file_path}")
        mod_spec = importlib.util.spec_from_file_location("_marisco_local_cb", file_path)
        mod = importlib.util.module_from_spec(mod_spec)
        mod_spec.loader.exec_module(mod)
        if not hasattr(mod, self.class_):
            raise AttributeError(f"Class '{self.class_}' not found in {file_path}")
        return getattr(mod, self.class_)


PluginSpecModel = Annotated[
    Union[LegacyPathPluginSpec, NamePluginSpec, FilePluginSpec],
    Field(union_mode="smart"),
]
_PLUGIN_SPEC_ADAPTER = TypeAdapter(PluginSpecModel)


class PluginSpec:
    "Backward-compatible factory for polymorphic plugin specs."

    def __new__(cls, *args, **kwargs):
        data = args[0] if args else kwargs
        return _PLUGIN_SPEC_ADAPTER.validate_python(data)

    @classmethod
    def model_validate(cls, data):
        return _PLUGIN_SPEC_ADAPTER.validate_python(data)


class MeltEntry(BaseModel):
    "One wide-to-long mapping entry: value column, uncertainty column, nuclide, unit, and lab."
    val: str
    unc: str
    nuclide: str
    unit: str
    lab: str


class UnitConversionCfg(BaseModel):
    "Single unit-conversion rule with its physical factor and optional metadata."
    nuclide: str
    src_unit: str
    dst_unit: str
    factor: float
    factor_name: str = ""
    comment: str = ""


class HandlerConfig(BaseModel):
    "Complete handler configuration loaded from a YAML data-contract file."
    module_name: str
    title: str = ""
    description: str = ""
    url: str
    fname_out: str
    zenodo_id: str = ""
    fmt: str = "csv"
    rename: dict[str, str] = Field(default_factory=dict)
    string_cast: list[str] = Field(default_factory=list)
    columns: dict[str, str] = Field(default_factory=dict)
    normalize_case: dict[str, str] = Field(default_factory=dict)
    col_date: Optional[str] = None
    col_time: Optional[str] = None
    dt_format: str = "%Y-%m-%d"
    time_format: Optional[str] = None
    meta_cols: list[str] = Field(default_factory=list)
    melt_spec: list[MeltEntry] = Field(default_factory=list)
    unit_conversions: list[UnitConversionCfg] = Field(default_factory=list)
    nuclide_lut: dict[str, int] = Field(default_factory=dict)
    unit_lut: dict[str, int] = Field(default_factory=dict)
    lab_lut: dict[str, int] = Field(default_factory=dict)
    lab_constants: dict[str, str] = Field(default_factory=dict)
    area_default: int = 0
    keywords: list[str] = Field(default_factory=list)
    global_attrs: dict[str, str] = Field(default_factory=dict)
    loader: Optional[PluginSpecModel] = None

    @computed_field
    @property
    def mapped_columns(self) -> frozenset[str]:
        return frozenset({*self.columns.values(), *self.rename.values()})

    @computed_field
    @property
    def melt_columns(self) -> frozenset[str]:
        return frozenset(_MELT_PROVIDES) if self.melt_spec else frozenset()

    @computed_field
    @property
    def datetime_columns(self) -> frozenset[str]:
        return frozenset({"TIME"}) if self.col_date else frozenset()

    @computed_field
    @property
    def available_columns(self) -> frozenset[str]:
        return self.mapped_columns | self.melt_columns | self.datetime_columns

    @computed_field
    @property
    def missing_required_columns(self) -> frozenset[str]:
        return frozenset(_MARIS_REQUIRED - self.available_columns)

    @classmethod
    def from_yaml(cls, path: str | Path) -> "HandlerConfig":
        "Load and validate a handler YAML config with Gate 1 static quarantine."
        from marisco.handlers.pipeline.gates import load_handler_config
        return load_handler_config(cls, path)


_MARIS_REQUIRED = frozenset({"LAT", "LON", "TIME", "NUCLIDE", "VALUE", "UNC", "UNIT"})
_MELT_PROVIDES = frozenset({"NUCLIDE", "VALUE", "UNC", "UNIT"})


def ensure_known_global_attrs(attrs: dict[str, Any]) -> dict[str, Any]:
    "Validate NetCDF global attribute keys against the MARIS template vocabulary."
    unknown = set(attrs.keys()) - NC_GLOBAL_ATTRS
    if unknown:
        raise KeyError(
            f"Unknown NetCDF global attribute(s): {', '.join(sorted(unknown))}. "
            "Add to NC_GLOBAL_ATTRS in configs if intentional."
        )
    return attrs


In [ ]:
cfg = HandlerConfig.from_yaml("config/handlers/fram_strait.yaml")
print(cfg.title)
print(sorted(cfg.missing_required_columns))